# Listen & compare: ORIG vs model reconstructions

Loads the checkpoints from `render_samples.CKPTS` (applies EMA weights when present),
reconstructs 3 clips of the cookie song, then:
1. **verifies which weights are loaded** (EMA vs raw),
2. audio players,
3. waveform plots (full + zoom),
4. mel-spectrogram comparison (orig / recon / |diff|).

In [ ]:
import os
import torch, torchaudio
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from render_samples import CKPTS, load_module, pick_clips, reconstruct, SR, SLICE
from train import (
    build_learning_params, build_optimizer_cfg, build_scheduler_cfg, build_loss_aggregator,
)

lp, oc, sc = build_learning_params(), build_optimizer_cfg(), build_scheduler_cfg()
la = build_loss_aggregator()

clips = pick_clips(3)  # [(name, [1, L])], 25/50/75% of the song
modules = {}
for tag, cfg in CKPTS.items():
    if not os.path.exists(cfg["path"]):
        print(f"SKIP {tag}: {cfg['path']} missing")
        continue
    print(f"Loading {tag} <- {cfg['path']}")
    modules[tag] = load_module(
        cfg["path"], lp, oc, sc, la,
        token_dim=cfg["token_dim"], latent_grid=cfg["latent_grid"],
        bypass_vq=cfg["bypass_vq"], arch=cfg["arch"],
    )

with torch.no_grad():
    recons = {tag: [reconstruct(m, clip) for _, clip in clips] for tag, m in modules.items()}
print("done:", list(recons), "x", len(clips), "clips")

## 1. Which weights are actually loaded?

Relative difference between the loaded generator params and the **raw** `state_dict` in the ckpt.
`rel-diff = 0` would mean raw weights; `> 0` means the EMA copy was applied on top.
Also reports how far EMA is from raw per ckpt (drift of the smoothed model).

In [ ]:
for tag in modules:
    ck = torch.load(CKPTS[tag]["path"], map_location="cpu", weights_only=False)
    raw = {k: v for k, v in ck["state_dict"].items() if k.startswith("model.")}
    cur = dict(modules[tag].named_parameters())
    rels = []
    for k, v in raw.items():
        if k in cur and cur[k].shape == v.shape:
            rels.append(((cur[k].detach() - v).norm() / (v.norm() + 1e-12)).item())
    rels = np.array(rels)
    verdict = "RAW weights (EMA NOT applied!)" if rels.mean() < 1e-7 else "EMA applied (differs from raw)"
    print(f"{tag}: {len(rels)} generator tensors, mean rel-diff {rels.mean():.4f}, max {rels.max():.4f} -> {verdict}")

## 2. Listen

In [ ]:
def norm1d(x):
    x = x.flatten().float()
    return (x / (x.abs().max() + 1e-8)).numpy()

for i, (name, clip) in enumerate(clips):
    print(f"=== clip{i} ({name}) — ORIG ===")
    display(Audio(norm1d(clip), rate=SR))
    for tag in modules:
        print(f"--- {tag}")
        display(Audio(norm1d(recons[tag][i]), rate=SR))

## 3. Waveforms (full clip + 30 ms zoom)

Zoom window is mid-clip. Look for: amplitude envelope tracking (full view) and
whether the fine oscillation is periodic like the original or noise-like (zoom).

In [ ]:
ZOOM = int(0.030 * SR)  # 30 ms
rows = 1 + len(modules)
for i, (name, clip) in enumerate(clips):
    L = clip.shape[1]
    z0 = L // 2 - ZOOM // 2
    fig, axes = plt.subplots(rows, 2, figsize=(14, 2.2 * rows), sharex="col")
    sigs = [("ORIG", norm1d(clip))] + [(tag, norm1d(recons[tag][i])) for tag in modules]
    t_full = np.arange(L) / SR
    t_zoom = np.arange(ZOOM) / SR * 1000
    for r, (tag, y) in enumerate(sigs):
        axes[r, 0].plot(t_full, y, lw=0.3)
        axes[r, 0].set_ylabel(tag, fontsize=9)
        axes[r, 1].plot(t_zoom, y[z0 : z0 + ZOOM], lw=0.8)
    axes[0, 0].set_title(f"clip{i} full waveform")
    axes[0, 1].set_title("30 ms zoom (mid-clip)")
    axes[-1, 0].set_xlabel("s")
    axes[-1, 1].set_xlabel("ms")
    plt.tight_layout(); plt.show()

## 4. Mel spectrograms: ORIG / recon / |diff|

Log-mel, 128 bands. Diff column is |log-mel(orig) − log-mel(recon)| on a shared scale —
bright regions = where the reconstruction misses energy structure (harmonic stacks =
melody; vertical stripes = transients/rhythm).

In [ ]:
mel_t = torchaudio.transforms.MelSpectrogram(SR, n_fft=2048, hop_length=512, n_mels=128)

def logmel(x_1d):
    return torch.log10(mel_t(torch.as_tensor(x_1d, dtype=torch.float32)) + 1e-6).numpy()

for i, (name, clip) in enumerate(clips):
    lm_orig = logmel(norm1d(clip))
    fig, axes = plt.subplots(len(modules), 3, figsize=(15, 3.2 * len(modules)), squeeze=False)
    for r, tag in enumerate(modules):
        lm_rec = logmel(norm1d(recons[tag][i]))
        vmin, vmax = lm_orig.min(), lm_orig.max()
        for c, (ttl, img, kw) in enumerate([
            (f"ORIG (clip{i})", lm_orig, dict(vmin=vmin, vmax=vmax, cmap="magma")),
            (tag, lm_rec, dict(vmin=vmin, vmax=vmax, cmap="magma")),
            (f"|diff| {tag}", np.abs(lm_orig - lm_rec), dict(vmin=0, vmax=2.0, cmap="viridis")),
        ]):
            im = axes[r][c].imshow(img, origin="lower", aspect="auto", **kw)
            axes[r][c].set_title(ttl, fontsize=9)
            fig.colorbar(im, ax=axes[r][c], fraction=0.046)
    plt.tight_layout(); plt.show()

### Reading guide
- **Weight check (cell 1)**: if any model prints `RAW weights (EMA NOT applied!)`, the render/listen used the wrong weights — report back.
- **Zoomed waveform**: original shows clean periodic oscillation; if recon looks like dense noise, phase is incoherent (the known melody-killer).
- **Mel diff**: horizontal bright bands across time = missing/blurred harmonics (pitch); short bright verticals = smeared transients.